In [54]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [1, 10]
selected_campaigns = list(range(1, 21))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        rf_param.value,
    ],
    campaigns=selected_campaigns,
)


Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [55]:
import pandas as pd
from scipy.spatial.distance import cdist


def compute_weights_with_best_rps(
        m_rfp: np.array,
        idx_rfp: np.array,
        m_tp: np.array,
        idx_tp: np.array,
        df_tp: pd.DataFrame = None
) -> (np.array, np.array):
    """
    Computes weights for two matrices with a single reference point parameter.
    :param m_rfp: point matrix for the reference points (2D array)
    :param idx_rfp: valid index matrix for the reference points (1D array)
    :param m_tp: point matrix for the test points (2D array)
    :param idx_tp: valid index matrix for the test points (1D array)
    :return: Weights and sorted indices by weight
    """

    # Compute the Euclidean distances between the TPs and RPs
    D = cdist(m_tp, m_rfp, metric="euclidean")

    # Normalize distances based on common valid indices
    match = np.logical_and(idx_tp[:, np.newaxis, :], idx_rfp[np.newaxis, :, :])
    s = np.sum(match, axis=2)

    # Avoid division by zero by setting distances to a very large value where no matches exist
    realmax = np.finfo(np.float64).max
    D = np.divide(D, s, out=np.full_like(D, realmax), where=s != 0)

    # Set distances to dummy reference points to a very large value
    dummy_rfps = np.all(idx_rfp == 0, axis=1)
    D[:, dummy_rfps] = realmax

    # Replace zero distances with a small value to avoid singularities
    min_nonzero_distance = np.min(D[D > 0])
    D[D == 0] = min_nonzero_distance / 20

    # set non-optimal paths to max value
    if df_tp is not None:
        for i, row in df_tp.iterrows():
            matches = row['matches']
            non_matching_indices = set(range(m_rfp.shape[0])) - set(matches)
            D[i, list(non_matching_indices)] = realmax

    # Sort distances and compute weights
    idx_sort = np.argsort(D, axis=1)
    D_sort = np.take_along_axis(D, idx_sort, axis=1)
    W = 1.0 / D_sort

    return W, idx_sort


In [70]:

from scripts.data_processing import cluster_data_and_train_random_forest
from scripts.utils import extract_unique_npcis, dataset_tp_rp_split
from scripts.beamforming import get_best_beam, find_matching_rps

# iterate over tps
errors = []
unique_npcis = extract_unique_npcis(df["measurements_matrix"])

df = df.sample(100)
df['best_beam'] = df['measurements_matrix'].apply(lambda x: get_best_beam(x, rf_param))

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 42)

df_tp["matches"] = df_tp["best_beam"].apply(lambda x: find_matching_rps(df_rp, x))

rf_model = cluster_data_and_train_random_forest(
    df_rp, 5, unique_npcis, rf_param, 42
)








In [72]:
for name, group in df_rp.groupby("cluster"):
    print(name)
    print(group.index)

0
Index([3, 4, 9, 14, 17, 24, 25, 26, 35, 37, 38, 41, 42, 60, 61, 63], dtype='int64')
1
Index([0, 1, 5, 8, 12, 13, 20, 34, 40, 49, 50, 52, 55, 62, 64], dtype='int64')
2
Index([6, 10, 11, 15, 16, 18, 19, 21, 27, 29, 30, 45, 46, 48, 54, 58, 65], dtype='int64')
3
Index([7, 22, 33, 44, 51, 57, 59], dtype='int64')
4
Index([2, 23, 28, 31, 32, 36, 39, 43, 47, 53, 56], dtype='int64')


In [59]:

#
# m_rfp, idx_rfp = create_point_matrix(df_rp, unique_npcis, rf_param)
#
# # Create the point matrix for the test points
# m_tp, idx_tp = create_point_matrix(df_tp, unique_npcis, rf_param)
#
# W, idx_sort = compute_weights_with_best_rps(m_rfp, idx_rfp, m_tp, idx_tp, df_tp)
#
# W_base, idx_sort_base = compute_weights_with_best_rps(m_rfp, idx_rfp, m_tp, idx_tp)
#
# _, k_avg_error = wknn_one(df_tp, df_rp, idx_sort, W, 2)
#
# _, k_avg_error_base = wknn_one(df_tp, df_rp, idx_sort_base, W_base, 2)


array([[32, 34, 51, ..., 30, 64, 65],
       [20, 35, 31, ..., 64, 16, 65],
       [35,  9, 31, ..., 64, 16, 65],
       ...,
       [26, 38,  7, ..., 64, 18, 65],
       [ 5,  0, 61, ..., 31, 17, 65],
       [57, 22, 10, ..., 64, 34, 65]])

In [68]:
RFP_selected_idx = idx_sort[:, :2]

RFP_selected_idx.flatten()

array([32, 34, 20, 35, 35,  9, 23, 56, 20, 35, 54, 17, 53, 21, 47, 37,  0,
       35, 55, 13,  8, 17, 32, 34, 56, 23, 35,  9, 45, 13,  0, 35,  0, 35,
       17, 54, 32, 34, 58,  0, 23, 56, 62, 15, 58,  0, 41, 46, 29, 43, 14,
        2,  0, 35, 56, 23, 15, 62, 14,  2, 50, 12, 26, 38,  5,  0, 57, 22])

In [64]:
tp = df_tp.iloc[1]

n_matches = len(tp['matches'])

total = len(df_rp)

non_matching_indices = set(range(m_rfp.shape[0])) - set(tp["matches"])
n_non_matching = len(non_matching_indices)
print(f"""
matches: {n_matches}
total: {total}
non_matching: {n_non_matching}
""")

print(tp['matches'])


matches: 5
total: 66
non_matching: 61

Index([9, 20, 28, 31, 35], dtype='int64')
